## Load GTFS Reference Data into TrainAnalysis KQL Database

Downloads Timetables For Realtime v2 GTFS bundle from Transport NSW,
extracts stops.txt, routes.txt, and stop_times.txt, and loads them
as reference tables into the TrainAnalysis KQL database.

**Run cadence:** Once initially, then re-run when timetables change.

In [ ]:
pip install requests pandas azure-kusto-data azure-kusto-ingest

In [ ]:
import requests
import pandas as pd
import zipfile
import io
import json
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.ingest import (
    KustoStreamingIngestClient,
    IngestionProperties,
    DataFormat,
)
from datetime import datetime
import pytz

In [ ]:
# Transport NSW API key
myapikey = ""

# TrainAnalysis KQL database Query URI
# Find in Fabric portal: open TrainAnalysis Eventhouse -> copy Query URI
kusto_uri = ""

# KQL database name
database_name = "TrainAnalysis"

In [ ]:
def download_gtfs_bundle(api_key):
    """Download the GTFS static timetable bundle for Sydney Trains."""
    url = "https://api.transport.nsw.gov.au/v1/gtfs/schedule/sydneytrains"
    headers = {
        "Authorization": f"apikey {api_key}",
        "Accept": "application/zip"
    }
    print("Downloading GTFS bundle from Transport NSW...")
    response = requests.get(url, headers=headers, timeout=120)
    response.raise_for_status()
    zip_bytes = io.BytesIO(response.content)
    gtfs_zip = zipfile.ZipFile(zip_bytes)
    file_list = gtfs_zip.namelist()
    print(f"Downloaded bundle with {len(file_list)} files:")
    for name in sorted(file_list):
        info = gtfs_zip.getinfo(name)
        print(f"  {name} ({info.file_size:,} bytes)")
    return gtfs_zip

gtfs_zip = download_gtfs_bundle(myapikey)

In [ ]:
def parse_gtfs_file(gtfs_zip, filename, usecols=None):
    """Parse a CSV file from the GTFS ZIP into a DataFrame."""
    with gtfs_zip.open(filename) as f:
        df = pd.read_csv(f, dtype=str, usecols=usecols)
    print(f"{filename}: {len(df):,} rows, columns: {list(df.columns)}")
    return df

# stops.txt - stop_id to station name mapping
df_stops = parse_gtfs_file(gtfs_zip, "stops.txt", usecols=[
    "stop_id", "stop_name", "stop_lat", "stop_lon",
    "location_type", "parent_station", "platform_code"
])
df_stops["stop_lat"] = pd.to_numeric(df_stops["stop_lat"], errors="coerce")
df_stops["stop_lon"] = pd.to_numeric(df_stops["stop_lon"], errors="coerce")

# routes.txt - route_id to line name mapping
df_routes = parse_gtfs_file(gtfs_zip, "routes.txt", usecols=[
    "route_id", "route_short_name", "route_long_name",
    "route_type", "route_color", "route_text_color"
])

# stop_times.txt - scheduled arrival/departure times per trip per stop
df_stop_times = parse_gtfs_file(gtfs_zip, "stop_times.txt", usecols=[
    "trip_id", "arrival_time", "departure_time",
    "stop_id", "stop_sequence"
])
df_stop_times["stop_sequence"] = pd.to_numeric(
    df_stop_times["stop_sequence"], errors="coerce"
).astype("Int64")

print("\nPreview - stops:")
print(df_stops.head())
print("\nPreview - routes:")
print(df_routes.head())

In [ ]:
# Authenticate to KQL database
try:
    token = mssparkutils.credentials.getToken("https://kusto.kusto.windows.net")
    print("Authenticated via Fabric notebook identity")
except NameError:
    import subprocess
    result = subprocess.run(
        ["az", "account", "get-access-token", "--resource",
         "https://kusto.kusto.windows.net", "--query", "accessToken", "-o", "tsv"],
        capture_output=True, text=True
    )
    token = result.stdout.strip()
    print("Authenticated via Azure CLI")

kcsb = KustoConnectionStringBuilder.with_aad_user_token_authentication(kusto_uri, token)
kusto_client = KustoClient(kcsb)
test_result = kusto_client.execute(database_name, ".show database schema | count")
print(f"Connected to {database_name} successfully")

In [ ]:
# Create table schemas (idempotent - create-merge won't fail if exists)
table_commands = [
    ".create-merge table StopsReference (stop_id: string, stop_name: string, stop_lat: real, stop_lon: real, location_type: string, parent_station: string, platform_code: string)",
    ".create-merge table RoutesReference (route_id: string, route_short_name: string, route_long_name: string, route_type: string, route_color: string, route_text_color: string)",
    ".create-merge table StopTimesReference (trip_id: string, arrival_time: string, departure_time: string, stop_id: string, stop_sequence: int)"
]
for cmd in table_commands:
    kusto_client.execute_mgmt(database_name, cmd)
print("Table schemas created/verified")

In [ ]:
def ingest_dataframe(kusto_client, kcsb, database_name, table_name, df, batch_size=10000):
    """Ingest a DataFrame into a KQL table using streaming ingestion."""
    total_rows = len(df)
    print(f"\nClearing existing {table_name} data...")
    kusto_client.execute_mgmt(database_name, f".clear table {table_name} data")
    
    ingest_client = KustoStreamingIngestClient(kcsb)
    props = IngestionProperties(
        database=database_name,
        table=table_name,
        data_format=DataFormat.CSV,
    )
    
    ingested = 0
    for start in range(0, total_rows, batch_size):
        batch = df.iloc[start:start + batch_size]
        csv_data = batch.to_csv(index=False, header=False)
        stream = io.BytesIO(csv_data.encode("utf-8"))
        ingest_client.ingest_from_stream(stream, ingestion_properties=props)
        ingested += len(batch)
        print(f"  {table_name}: {ingested:,}/{total_rows:,} rows ingested")
    
    result = kusto_client.execute(database_name, f"{table_name} | count")
    count = [row[0] for row in result.primary_results[0]][0]
    print(f"  Verified: {table_name} has {count:,} rows")
    return count

In [ ]:
print("=" * 60)
print("Ingesting reference data into TrainAnalysis KQL database")
print("=" * 60)

ingest_dataframe(kusto_client, kcsb, database_name, "StopsReference", df_stops, batch_size=10000)
ingest_dataframe(kusto_client, kcsb, database_name, "RoutesReference", df_routes, batch_size=10000)

print("\nIngesting StopTimesReference (this may take a few minutes)...")
ingest_dataframe(kusto_client, kcsb, database_name, "StopTimesReference", df_stop_times, batch_size=50000)

print("\n" + "=" * 60)
print("Reference data load complete!")
print("=" * 60)

In [ ]:
# Verify reference data
queries = [
    ("Central Station stops", 'StopsReference | where stop_name contains "Central" | take 5'),
    ("Train routes", "RoutesReference | take 10"),
    ("Sample stop times", "StopTimesReference | take 5"),
]
for label, query in queries:
    print(f"\n--- {label} ---")
    result = kusto_client.execute(database_name, query)
    cols = [c.column_name for c in result.primary_results[0].columns]
    for row in result.primary_results[0]:
        print(dict(zip(cols, row)))